# Introduction to Action-Angle Coordinates

Action-angle coordinates $(\mathbf{J}, \boldsymbol{\theta})$ are canonical
coordinates for integrable Hamiltonian systems in which the actions $\mathbf{J}$
are constants of motion and the angles $\boldsymbol{\theta}$ increase linearly
with time at rates given by the frequencies $\boldsymbol{\Omega}$.

For an axisymmetric potential, the three actions are:
- $J_R$: radial action (related to the radial oscillation amplitude)
- $L_z$: z-component of angular momentum (azimuthal action)
- $J_z$: vertical action (related to the vertical oscillation amplitude)

The corresponding frequencies are $\Omega_R$, $\Omega_\phi$, and $\Omega_z$.

galpy supports both **forward** transformations $(\mathbf{x}, \mathbf{v}) \to (\mathbf{J}, \boldsymbol{\theta}, \boldsymbol{\Omega})$
and **reverse** transformations $(\mathbf{J}, \boldsymbol{\theta}) \to (\mathbf{x}, \mathbf{v})$.

In [ ]:
%matplotlib inline
import numpy
import matplotlib.pyplot as plt

## Using the Orbit interface

The simplest way to compute actions, frequencies, and angles is through
the `Orbit` class. After integrating an orbit, you can call methods like
`o.jr()`, `o.jp()`, `o.jz()` for actions, `o.Or()`, `o.Op()`, `o.Oz()`
for frequencies, and `o.wr()`, `o.wp()`, `o.wz()` for angles.

These methods automatically select an appropriate action-angle calculator
based on the potential.

In [ ]:
from galpy.orbit import Orbit
from galpy.potential import MWPotential2014

o = Orbit([1.0, 0.1, 1.1, 0.0, 0.05, 0.0])
ts = numpy.linspace(0, 100, 10001)
o.integrate(ts, MWPotential2014)

# Actions
print("J_R  =", o.jr(pot=MWPotential2014))
print("L_z  =", o.jp(pot=MWPotential2014))
print("J_z  =", o.jz(pot=MWPotential2014))

In [ ]:
# Frequencies
print("Omega_R   =", o.Or(pot=MWPotential2014))
print("Omega_phi =", o.Op(pot=MWPotential2014))
print("Omega_z   =", o.Oz(pot=MWPotential2014))

In [ ]:
# Angles (at the initial time)
print("theta_R   =", o.wr(pot=MWPotential2014))
print("theta_phi =", o.wp(pot=MWPotential2014))
print("theta_z   =", o.wz(pot=MWPotential2014))

## Isochrone potential: exact action-angle coordinates

The isochrone potential is special because it has exact analytical
action-angle coordinates. This makes it ideal for testing.

In [ ]:
from galpy.potential import IsochronePotential
from galpy.actionAngle import actionAngleIsochrone

ip = IsochronePotential(b=0.9, normalize=1.0)
aAI = actionAngleIsochrone(ip=ip)

### Direct usage of actionAngleIsochrone

We can compute actions, frequencies, and angles by passing
phase-space coordinates directly to the `actionsFreqsAngles` method:

In [ ]:
# Phase-space point: R, vR, vT, z, vz, phi
R, vR, vT, z, vz, phi = 1.0, 0.1, 1.1, 0.0, 0.05, 0.0

jr, jphi, jz, Or, Op, Oz, wr, wp, wz = aAI.actionsFreqsAngles(R, vR, vT, z, vz, phi)
print(f"J_R = {jr[0]:.6f}")
print(f"L_z = {jphi[0]:.6f}")
print(f"J_z = {jz[0]:.6f}")
print(f"Omega_R = {Or[0]:.6f}, Omega_phi = {Op[0]:.6f}, Omega_z = {Oz[0]:.6f}")
print(f"theta_R = {wr[0]:.6f}, theta_phi = {wp[0]:.6f}, theta_z = {wz[0]:.6f}")

If you only need the actions (not the frequencies or angles), use the
`__call__` method which is faster:

In [ ]:
jr, jphi, jz = aAI(R, vR, vT, z, vz, phi)
print(f"J_R = {jr[0]:.6f}, L_z = {jphi[0]:.6f}, J_z = {jz[0]:.6f}")

### Verifying action conservation along an orbit

Actions should be conserved along an orbit. Let's integrate an orbit
in the isochrone potential and verify this:

In [ ]:
o = Orbit([1.0, 0.1, 1.1, 0.0, 0.05, 0.0])
ts = numpy.linspace(0.0, 50.0, 1001)
o.integrate(ts, ip)

# Compute actions at each timestep
Rs = o.R(ts)
vRs = o.vR(ts)
vTs = o.vT(ts)
zs = o.z(ts)
vzs = o.vz(ts)
phis = o.phi(ts)

jrs, jphis, jzs = aAI(Rs, vRs, vTs, zs, vzs, phis)

plt.plot(ts, jrs, label=r"$J_R$")
plt.plot(ts, jzs, label=r"$J_z$")
plt.xlabel(r"$t$")
plt.ylabel(r"$J$")
plt.legend()
plt.title("Action conservation in the isochrone potential")

The actions are constant to machine precision, confirming the exactness
of the isochrone action-angle calculation.

## Spherical potentials: actionAngleSpherical

For any spherical potential, actions can be computed exactly using
`actionAngleSpherical`, which uses numerical integration of the radial
and vertical oscillations:

In [ ]:
from galpy.potential import LogarithmicHaloPotential
from galpy.actionAngle import actionAngleSpherical

# Spherical logarithmic halo (q=1)
lp = LogarithmicHaloPotential(normalize=1.0, q=1.0)
aAS = actionAngleSpherical(pot=lp)

jr, jphi, jz = aAS(1.0, 0.1, 1.1, 0.0, 0.05, 0.0)
print(f"J_R = {jr[0]:.6f}, L_z = {jphi[0]:.6f}, J_z = {jz[0]:.6f}")

In [ ]:
# Full actions, frequencies, and angles
jr, jphi, jz, Or, Op, Oz, wr, wp, wz = aAS.actionsFreqsAngles(
    1.0, 0.1, 1.1, 0.0, 0.05, 0.0
)
print(f"Omega_R = {Or[0]:.4f}, Omega_phi = {Op[0]:.4f}, Omega_z = {Oz[0]:.4f}")